In [1]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from collections import Counter
import pickle
from pathlib import Path

from LLMGen import make_seq2seq_pairs, reduce_input_block, make_seq2seq_pairs_reduced
from LLMGen import filter_cases_by_length, sample_few_shots
from LLMGen import assemble_prompt_from_fewshots, generate_traces_for_batch, llm_results_to_eventlog, gt_add_col, get_bucket_size, compute_bucket_k
from LLMGen import save_checkpoint, load_checkpoint, save_pickle, load_pickle, append_df_to_csv

from Evaluation import evaluate_comprehensive, evaluate_light
from ollama import Client

In [2]:
#dataname = "helpdesk"
#dataname = "sepsis"
#dataname = "BPI13I"
#dataname = "BPI13C"
#dataname = "BPI20"
#dataname = "BPI12W"
#dataname = "BPI12"
#dataname = "BPI17"
dataname = "BPI20R"

In [3]:
context_size = 4096

Reuse the previous train-test-split datasets as if runing the second LLM model

In [4]:
train_event = pd.read_csv("../output/data_processed/" + dataname + "_train.csv")
test_event =  pd.read_csv("../output/data_processed/" + dataname + "_hold.csv")

Prepare for LLM input

In [5]:
# Rename all columns
def rename(event, timecol, sequence_id):
    event = event.copy()
    event[timecol] = pd.to_datetime(event[timecol])
    event = event.sort_values([sequence_id, timecol])
    event.columns = [c.replace(":", "_") for c in event.columns]
    return event

train_event = rename(train_event, "time:timestamp", "case:concept:name")
test_event = rename(test_event,  "time:timestamp", "case:concept:name")

In [6]:
case_index = 'case_concept_name'
time_col = 'time_timestamp'
core_event = "concept_name"
delta_col='delta_time'

In [7]:
if dataname == "helpdesk":
    cat_cols_event = ['org_resource']
    num_cols_event = []
    cat_cols_seq = ['case_variant']
    num_cols_seq = []
elif dataname == "BPI12" or dataname == "BPI12W":
    train_event[case_index] = train_event[case_index].astype(str)
    test_event[case_index] = test_event[case_index].astype(str)
    #train_df[case_index] = train_df[case_index].astype(str)
    #val_df[case_index] = val_df[case_index].astype(str)
    cat_cols_event = ['org_resource']
    num_cols_event = []
    cat_cols_seq = []
    num_cols_seq = ['case_AMOUNT_REQ']
elif dataname == "BPI13I" or dataname == "BPI13C":
    cat_cols_event = ['org_group', "resource country", "org_resource", "organization involved", "org_role"]
    num_cols_event = []
    cat_cols_seq = ["organization country", "impact", "product"]
    num_cols_seq = []
elif dataname == "sepsis":
    cat_cols_event = ['org_group']
    num_cols_event = ['Leucocytes', 'CRP', 'LacticAcid']
    cat_cols_seq = ['InfectionSuspected', 'DiagnosticBlood',     'DisfuncOrg',  'SIRSCritTachypnea', 'Hypotensie',       'SIRSCritHeartRate', 
                    'Infusion',           'DiagnosticArtAstrup', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther',
                    'SIRSCriteria2OrMore', 'DiagnosticXthorax',  'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 
                    'DiagnosticLacticAcid', 'Diagnose',          'Hypoxie',             'DiagnosticUrinarySediment', 'DiagnosticECG']
    num_cols_seq = [ 'Age']
elif dataname == "BPI17":
    cat_cols_event = ['Action', 'org_resource', 'EventOrigin', 'Accepted', 'Selected', "OfferID"]
    num_cols_event = ['FirstWithdrawalAmount', 'NumberOfTerms', 'MonthlyCost',  'CreditScore', 'OfferedAmount']
    cat_cols_seq = [ 'case_LoanGoal', 'case_ApplicationType']
    num_cols_seq = ['case_RequestedAmount']
elif dataname == "BPI20":
    cat_cols_event = ['org_role']
    num_cols_event = []
    cat_cols_seq = ["case_OrganizationalEntity", "case_Project"]
    num_cols_seq = ["case_RequestedAmount", "case_Permit RequestedBudget"]  
elif dataname == "BPI20R":
    cat_cols_event = ['org_resource','org_role']
    num_cols_event = []
    cat_cols_seq = ["case_OrganizationalEntity", "case_Project", "case_RfpNumber", "case_Task", "case_Activity"]
    num_cols_seq = ["case_RequestedAmount"]  
    
attr_cols = cat_cols_event + num_cols_event + cat_cols_seq + num_cols_seq

In [8]:
model = "llama3.1:8b-instruct-q5_K_M"
#model = "mistral:7b-instruct-q5_K_M"

In [9]:
# this version has limited metadata
if model == "llama3.1:8b-instruct-q5_K_M" and context_size == 32768:
    train_pairs = make_seq2seq_pairs_reduced(event = train_event,
                                 case_id_col = case_index,
                                 timestamp_col = time_col,
                                 activity_col = core_event,
                                 # event-level attributes by type
                                 event_attr_cols_cat = cat_cols_event,
                                 event_attr_cols_num = num_cols_event,
                                 event_attr_cols_bol = None,
                                 # sequence-level attributes by type
                                 sequence_attr_cols_cat = cat_cols_seq,
                                 sequence_attr_cols_num = num_cols_seq,
                                 sequence_attr_cols_bol = None,
                                 output_file = "../output/data_processed/" + dataname +"_train_seq2seq_reduced",
                                 return_pairs = True
                                )

    test_pairs = make_seq2seq_pairs_reduced(event = test_event,
                                 case_id_col = case_index,
                                 timestamp_col = time_col,
                                 activity_col = core_event,
                                 # event-level attributes by type
                                 event_attr_cols_cat = cat_cols_event,
                                 event_attr_cols_num = num_cols_event,
                                 event_attr_cols_bol = None,
                                 # sequence-level attributes by type
                                 sequence_attr_cols_cat = cat_cols_seq,
                                 sequence_attr_cols_num = num_cols_seq,
                                 sequence_attr_cols_bol = None,
                                 output_file = "../output/data_processed/" + dataname + "_hold_seq2seq_reduced",
                                 return_pairs = True
                                )
else:
    # Reuse the previous stored pairs for the second model
    with open("../output/data_processed/" + dataname + "_train_seq2seq_reduced.pkl", "rb") as f:
        train_pairs  = pickle.load(f)
    
    with open("../output/data_processed/" + dataname + "_hold_seq2seq_reduced.pkl", "rb") as f:
        test_pairs  = pickle.load(f)
        

Saved seq2seq training file to: ..\output\data_processed\BPI20R_train_seq2seq_reduced.txt
Saved seq2seq training file to: ..\output\data_processed\BPI20R_hold_seq2seq_reduced.txt


Manually define the length bucket for generation according to the range of length of holdout dataset

In [10]:
from collections import Counter

def get_case_length_counter(df, case_index):
    """
    Return Counter of case lengths from a pandas event log DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Event log in long format, one row per event
    case_index : str
        Column name for case id

    Returns
    -------
    Counter
        Example: Counter({4: 2267, 5: 938, ...})
    """
    case_lengths = df.groupby(case_index).size()
    return Counter(case_lengths.tolist())

In [11]:
get_case_length_counter(train_event, 'case_concept_name')

Counter({5: 3030,
         6: 1618,
         4: 580,
         3: 371,
         8: 200,
         9: 153,
         1: 72,
         10: 56,
         7: 47,
         11: 34,
         12: 13,
         2: 8,
         13: 6,
         15: 4,
         16: 3,
         14: 2,
         20: 1})

In [22]:
# helpdesk dataset
if dataname == "helpdesk":
    length_buckets = {
        "short":  (2, 5),   # 271 traces
        "medium": (5, 7),   # 154 traces
        "long":   (7, None) # 32 traces (7–11 merged)
    }
elif dataname == "sepsis":
    length_buckets = {
    "short":  (3, 8),
    "medium": (8, 18),
    "long":   (18, None)
    }
elif dataname == "BPI13I":
    length_buckets = {
    "short":  (1, 6),    # 1–5
    "medium": (6, 16),   # 6–15
    "long":   (16, None) # 16+
    }
elif dataname == "BPI13C":
    length_buckets = {
    "short":  (1, 4),    # 1–3
    "medium": (4, 8),    # 4–7
    "long":   (8, None)  # 8+
}
elif dataname == "BPI20":
    length_buckets = {
    "short":  (1, 8),    # 1–7
    "medium": (8, 12),   # 8–11
    "long":   (12, None) # 12+
    }
elif dataname == "BPI12W":
    length_buckets = {
    "short":  (2, 8),     # 2–7
    "medium": (8, 20),    # 8–19
    "long":   (20, 46),   # 20–45
    "xlong":  (46, None)  # 46+
}
elif dataname == "BPI12":
    length_buckets = {
        "short":  (3, 8),     # 3–7
        "medium": (8, 24),    # 8–23
        "long":   (24, 61),   # 24–60
        "xlong":  (61, None)  # 61+
    }
elif dataname == "BPI17":
    length_buckets = {
    "short":  (10, 21),    # 10–20
    "medium": (21, 41),    # 21–40
    "long":   (41, 61),    # 41–60
    "xlong":  (61, None)   # 61+
}
elif dataname == "BPI20R":
    length_buckets = {
    "short":  (1, 5),
    "medium": (5, 7),
    "long":   (7, None)
}

Generate Traces

In [10]:
all_generated = []

client = Client()

In [13]:
if model == "llama3.1:8b-instruct-q5_K_M" and context_size == 32768:
    model_tag = "llama_32"
elif model == "llama3.1:8b-instruct-q5_K_M" and context_size == 4096:
    model_tag = "llama"    
elif model == "mistral:7b-instruct-q5_K_M" and context_size == 4096:
    model_tag = "mistral"
   
checkpoint_path = f"../output/runs/{dataname}_{model_tag}_checkpoint.json"
parsed_csv_path = f"../output/runs/{dataname}_{model_tag}_parsed_rows.csv"
fewshots_dir = Path(f"../output/runs/{dataname}_{model_tag}_fewshots")
raw_jsonl_path = f"../output/runs/{dataname}_{model_tag}_raw_generations.jsonl"

# =========================================================
# SETTINGS
# =========================================================
save_every_n = 10


# =========================================================
# CHECKPOINT
# =========================================================
checkpoint = load_checkpoint(checkpoint_path)

if checkpoint is None:
    checkpoint = {
        "bucket_idx": 0,
        "case_idx": 0
    }

bucket_items = list(length_buckets.items())

# =========================================================
# MAIN LOOP
# =========================================================
for bucket_idx, (bucket_name, (min_len, max_len)) in enumerate(bucket_items):

    if bucket_idx < checkpoint["bucket_idx"]:
        continue

    print(f"\n=== Bucket: {bucket_name} ({min_len}, {max_len}) ===")

    bucket_size = get_bucket_size(train_pairs, min_len, max_len)
    k = compute_bucket_k(bucket_size, min_k=3, max_k=10, scale=0.4)
    print(f"{bucket_name}: bucket_size={bucket_size}, k={k}")

    # exact same few-shots on resume
    fewshot_path = fewshots_dir / f"{bucket_name}.pkl"
    few_shots = load_pickle(fewshot_path)

    if few_shots is None:
        few_shots = sample_few_shots(
            train_pairs,
            k,
            min_len=min_len,
            max_len=max_len,
            max_expand=5
        )
        save_pickle(few_shots, fewshot_path)

    if len(few_shots) == 0:
        print("[SKIP] No few-shot examples available.")
        checkpoint["bucket_idx"] = bucket_idx + 1
        checkpoint["case_idx"] = 0
        save_checkpoint(checkpoint, checkpoint_path)
        continue

    selected_holdouts = filter_cases_by_length(
        test_pairs,
        min_len=min_len,
        max_len=max_len
    )

    if len(selected_holdouts) == 0:
        print("[SKIP] No holdout cases in this bucket.")
        checkpoint["bucket_idx"] = bucket_idx + 1
        checkpoint["case_idx"] = 0
        save_checkpoint(checkpoint, checkpoint_path)
        continue

    selected_holdouts_inputs = [reduce_input_block(p) for p in selected_holdouts]

    start_case_idx = checkpoint["case_idx"] if bucket_idx == checkpoint["bucket_idx"] else 0

    all_generated = []

    for case_idx in range(start_case_idx, len(selected_holdouts_inputs)):
        new_input = selected_holdouts_inputs[case_idx]

        print(f"[RUN] bucket={bucket_name}, case_idx={case_idx}")

        result = generate_traces_for_batch([new_input], few_shots, model, client, context_size=context_size)[0]
        result["length_bucket"] = bucket_name
        result["bucket_case_idx"] = case_idx
        
        all_generated.append(result)

        if len(all_generated) >= save_every_n:
            
            with open(raw_jsonl_path, "a", encoding="utf-8") as f:
               for item in all_generated:
                   f.write(json.dumps(item, ensure_ascii=False) + "\n")            
            gen_df = llm_results_to_eventlog(
                all_generated,
                case_id_key=case_index,
                activity_key=core_event,
                time_key=time_col,
                attr_keys=attr_cols
            )


            if len(gen_df) > 0:
                append_df_to_csv(gen_df, parsed_csv_path)

            all_generated = []

            checkpoint["bucket_idx"] = bucket_idx
            checkpoint["case_idx"] = case_idx + 1
            save_checkpoint(checkpoint, checkpoint_path)

    # flush remaining cases in this bucket
    if len(all_generated) > 0:

        with open(raw_jsonl_path, "a", encoding="utf-8") as f:
            for item in all_generated:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        gen_df = llm_results_to_eventlog(
            all_generated,
            case_id_key=case_index,
            activity_key=core_event,
            time_key=time_col,
            attr_keys=attr_cols
        )

        if len(gen_df) > 0:
            append_df_to_csv(gen_df, parsed_csv_path)

        all_generated = []

    checkpoint["bucket_idx"] = bucket_idx + 1
    checkpoint["case_idx"] = 0
    save_checkpoint(checkpoint, checkpoint_path)

NameError: name 'model_tag' is not defined

In [11]:
if model == "llama3.1:8b-instruct-q5_K_M" and context_size == 32768:
    model_tag = "llama_32"
elif model == "llama3.1:8b-instruct-q5_K_M" and context_size == 4096:
    model_tag = "llama"    
elif model == "mistral:7b-instruct-q5_K_M" and context_size == 4096:
    model_tag = "mistral"
   
checkpoint_path = f"../output/runs/{dataname}_{model_tag}_checkpoint.json"
parsed_csv_path = f"../output/runs/{dataname}_{model_tag}_parsed_rows.csv"
fewshots_dir = Path(f"../output/runs/{dataname}_{model_tag}_fewshots")
gen_df = pd.read_csv(parsed_csv_path)

In [12]:
gen_df['model'] = model_tag

In [13]:
if model == "llama3.1:8b-instruct-q5_K_M" and context_size == 32768:
    gen_df.to_csv("../output/gen_traces/" + dataname +"_gen_llama_32.csv", index = False)
elif model == "llama3.1:8b-instruct-q5_K_M" and context_size == 4096:
    gen_df.to_csv("../output/gen_traces/" + dataname +"_gen_llama.csv", index = False)    
elif model == "mistral:7b-instruct-q5_K_M" and context_size == 4096:
    gen_df.to_csv("../output/gen_traces/" + dataname +"_gen_mixtral.csv", index = False)

In [14]:
#gen_df = pd.read_csv("../output/gen_traces/" + dataname + "_gen_llama_32.csv")
#gen_df = pd.read_csv("../output/gen_traces/" + dataname + "_gen_mixtral.csv")

In [15]:
print(gen_df.columns)

Index(['case_concept_name', 'pos', 'concept_name', 'time_timestamp',
       'length_bucket', 'org_resource', 'org_role',
       'case_OrganizationalEntity', 'case_Project', 'case_RfpNumber',
       'case_Task', 'case_Activity', 'case_RequestedAmount', 'model'],
      dtype='object')


In [16]:
# Rename all columns
def rename_reverse(event, timecol, sequence_id):
    event = event.copy()
    event[timecol] = pd.to_datetime(event[timecol], errors='coerce')  
    event = event.sort_values([sequence_id, timecol])
    event.columns = [c.replace("_", ":") for c in event.columns]
    return event

gen_df = rename_reverse(gen_df, "time_timestamp", "case_concept_name")
test_event = rename_reverse(test_event,  "time_timestamp", "case_concept_name")

In [17]:
if dataname == "helpdesk":
    cat_cols_event = ['org:resource']
    num_cols_event = []
    cat_cols_seq = ['case:variant']
    num_cols_seq = []
elif dataname == "BPI12" or dataname == "BPI12W":
    cat_cols_event = ['org:resource']
    num_cols_event = []
    cat_cols_seq = []
    num_cols_seq = ['case:AMOUNT:REQ']
elif dataname == "BPI13I" or dataname == "BPI13C":
    cat_cols_event = ['org:group', "resource country", "org:resource", "organization involved", "org:role"]
    num_cols_event = []
    cat_cols_seq = ["organization country", "impact", "product"]
    num_cols_seq = []
elif dataname == "sepsis":
    cat_cols_event = ['org:group']
    num_cols_event = ['Leucocytes', 'CRP', 'LacticAcid']
    cat_cols_seq = ['InfectionSuspected', 'DiagnosticBlood',     'DisfuncOrg',  'SIRSCritTachypnea', 'Hypotensie',       'SIRSCritHeartRate', 
                    'Infusion',           'DiagnosticArtAstrup', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther',
                    'SIRSCriteria2OrMore', 'DiagnosticXthorax',  'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 
                    'DiagnosticLacticAcid', 'Diagnose',          'Hypoxie',             'DiagnosticUrinarySediment', 'DiagnosticECG']
    num_cols_seq = [ 'Age']
elif dataname == "BPI17":
    cat_cols_event = ['Action', 'org:resource', 'EventOrigin', 'Accepted', 'Selected', "OfferID"]
    num_cols_event = ['FirstWithdrawalAmount', 'NumberOfTerms', 'MonthlyCost',  'CreditScore', 'OfferedAmount']
    cat_cols_seq = [ 'case:LoanGoal', 'case:ApplicationType']
    num_cols_seq = ['case:RequestedAmount']
elif dataname == "BPI20":
    cat_cols_event = ['org:role']
    num_cols_event = []
    cat_cols_seq = ["case:OrganizationalEntity", "case:Project"]
    num_cols_seq = ["case:RequestedAmount", "case:Permit RequestedBudget"]  
elif dataname == "BPI20R":
    cat_cols_event = ['org:resource','org:role']
    num_cols_event = []
    cat_cols_seq = ["case:OrganizationalEntity", "case:Project", "case:RfpNumber", "case:Task", "case:Activity"]
    num_cols_seq = ["case:RequestedAmount"]      
attr_cols = cat_cols_event + num_cols_event + cat_cols_seq + num_cols_seq

In [18]:
gen_df = gen_df.rename(columns={
    'length:bucket': 'length_bucket'  # Fix the bucket column name
})

In [19]:
#process ground truce trace to have the same column names with generative one
#gt_df = test_event[['case_concept_name', 'concept_name', 'time_timestamp', 'org_resource', 'case_variant']]
case_index = 'case:concept:name'
time_col = 'time:timestamp'
core_event = "concept:name"
gt_df = gt_add_col(test_event, length_buckets, case_index, time_col, 'pos', 'length_bucket')

Evaluation

In [20]:
evaluate_light(gen_df, gt_df, case_index, core_event, time_col, 'pos', name=model_tag, jsd_lambda=1.0)

,seq_coverage,bigram_jsd_unconditional,trace_similarity_unconditional,duration_wd_unconditional,dl_similarity_unconditional,bigram_jsd_conditional,trace_similarity_conditional,duration_wd_conditional,dl_similarity_conditional
llama,0.811047,0.643705,0.479823,739925.363885,0.46633,0.454751,0.591609,605680.712722,0.574974


In [24]:
def filter_by_length_bucket(df, bucket, case_id_col, bucket_col="length_bucket"):
    cases = df[df[bucket_col] == bucket][case_id_col].unique()
    return df[df[case_id_col].isin(cases)]

In [25]:
eval_buckets = {
    "all": None,          # full dataset
    **length_buckets      # short / medium / long
}

all_eval_dfs = []

bol_cols_event = []
bol_cols_seq = []

for bucket_name, _ in eval_buckets.items():

    if bucket_name == "all":
        gen_b = gen_df
        gt_b  = gt_df
    else:
        gen_b = filter_by_length_bucket(gen_df, bucket_name, case_index)
        gt_b  = filter_by_length_bucket(gt_df,  bucket_name, case_index)

    # Skip empty buckets
    if gen_b.empty or gt_b.empty:
        continue

    
    eval_df =evaluate_comprehensive(gen_b, gt_b, case_index, core_event, time_col, "pos", 
                       [core_event] + cat_cols_event, num_cols_event, bol_cols_event,
                       cat_cols_seq, num_cols_seq, bol_cols_seq,
                       name=model_tag, jsd_lambda=1.0
                      )

    # Add metadata columns
    eval_df["length_bucket"] = bucket_name
    eval_df["num_cases"] = gen_b[case_index].nunique()
    eval_df['data'] = dataname

    all_eval_dfs.append(eval_df)


In [26]:
final_eval_df = pd.concat(all_eval_dfs, axis=0).reset_index().rename(columns={"index": "model"})

In [27]:
final_eval_df = final_eval_df.round(4)

In [28]:
final_eval_df

,model,seq_coverage,bigram_jsd_unconditional,trace_similarity_unconditional,duration_wd_unconditional,dl_similarity_unconditional,bigram_jsd_conditional,trace_similarity_conditional,duration_wd_conditional,dl_similarity_conditional,...,seq_case:RfpNumber_inconsistent_pct,seq_case:Task_inconsistent_cases,seq_case:Task_inconsistent_pct,seq_case:Activity_inconsistent_cases,seq_case:Activity_inconsistent_pct,seq_case:RequestedAmount_inconsistent_cases,seq_case:RequestedAmount_inconsistent_pct,length_bucket,num_cases,data
0,llama,0.8110,0.6437,0.4798,7.399254e+05,0.4663,0.4548,0.5916,6.056807e+05,0.5750,...,0.0502,29,0.0520,29,0.0520,29,0.0520,all,558,BPI20R
1,llama,0.7241,0.8668,0.2766,3.597544e+05,0.2629,0.5909,0.3820,2.195315e+05,0.3631,...,0.1270,9,0.1429,9,0.1429,9,0.1429,short,63,BPI20R
2,llama,0.8154,0.5936,0.5259,6.924169e+05,0.5166,0.4090,0.6450,5.602546e+05,0.6335,...,0.0416,18,0.0416,18,0.0416,18,0.0416,medium,433,BPI20R
3,llama,0.8857,0.6802,0.3826,1.586786e+06,0.3381,0.5659,0.4320,1.448675e+06,0.3817,...,0.0323,2,0.0323,2,0.0323,2,0.0323,long,62,BPI20R


In [24]:
final_eval_df.to_csv("../output/metrics/" + dataname + "_eval_32.csv", index = False)

In [29]:
all_eval = pd.read_csv("../output/metrics/" + dataname+ "_eval.csv")
all_eval = pd.concat([all_eval, final_eval_df], ignore_index=True)
all_eval.to_csv("../output/metrics/" + dataname + "_eval.csv", index = False)

In [ ]:
# Before calling your evaluation function
def prepare_for_evaluation(gen_df, gt_df, time_col='time_timestamp'):
    """Quick fix for the hour=24 issue"""
    
    for df in [gen_df, gt_df]:
        if time_col in df.columns:
            # Convert to string
            df[time_col] = df[time_col].astype(str)
            
            # Fix the specific problem
            mask = df[time_col].str.contains('2012-01-12 24:25:46', na=False)
            if mask.any():
                print(f"Found {mask.sum()} problematic timestamps, fixing...")
                df.loc[mask, time_col] = '2012-01-13 00:25:46'
            
            # Convert to datetime
            df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    
    return gen_df, gt_df

# Use it
#gen_df, gt_df = prepare_for_evaluation(gen_df, gt_df)
#results = event_attribute_mae(gen_df, gt_df, case_id_col, event_attr_cols_num)

In [ ]:
# Build new_cases list for generation
selected_holdouts_inputs= [reduce_input_block(p) for p in selected_holdouts]
# In  case the test_pairs are generated from make_seq2seq_pairs_reduced (metadata already has being reduced) 
# selected_holdouts_inputs= [p["input"] for p in selected_holdouts] 

In [ ]:
llm_results_to_eventlog(results, attr_keys=["org_resource", "case_variant"])

In [ ]:
response = client.chat(model="phi3:mini", messages=[
    {"role": "user", "content": prompt}
])
print(response["message"]["content"])

In [ ]:
'''
# This is orginal enriched metadata inputs
# save the pairs in case it is the first time
train_pairs = make_seq2seq_pairs(event = train_event,
                                 case_id_col = "case_concept_name",
                                 timestamp_col = "time_timestamp",
                                 activity_col = "concept_name",
                                 # event-level attributes by type
                                 event_attr_cols_cat = ["org_resource"],
                                 event_attr_cols_num = None,
                                 event_attr_cols_bol = None,
                                 # sequence-level attributes by type
                                 sequence_attr_cols_cat = ["case_variant"],
                                 sequence_attr_cols_num = None,
                                 sequence_attr_cols_bol = None,
                                 #output_file = "D:/Research in UAE/llm/output/helpdesk_train_seq2seq",
                                 output_file = "../output/helpdesk_train_seq2seq",
                                 return_pairs = True
                                )

test_pairs = make_seq2seq_pairs(event = test_event,
                                 case_id_col = "case_concept_name",
                                 timestamp_col = "time_timestamp",
                                 activity_col = "concept_name",
                                 # event-level attributes by type
                                 event_attr_cols_cat = ["org_resource"],
                                 event_attr_cols_num = None,
                                 event_attr_cols_bol = None,
                                 # sequence-level attributes by type
                                 sequence_attr_cols_cat = ["case_variant"],
                                 sequence_attr_cols_num = None,
                                 sequence_attr_cols_bol = None,
                                 #output_file = "D:/Research in UAE/llm/output/helpdesk_hold_seq2seq",
                                 output_file = "../output/helpdesk_hold_seq2seq",
                                 return_pairs = True
                                )
''''